In [1]:
# =========================
# CELL 1 — Mount Drive
# =========================
from google.colab import drive
drive.mount("/content/drive")
print("✅ Drive mounted")


Mounted at /content/drive
✅ Drive mounted


In [2]:
# =========================
# CELL 2 — Install Ghostscript (Required by OCRmyPDF)
# =========================
!apt-get -y update
!apt-get -y install ghostscript


Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [83.8 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,341 kB]
Hit:9 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,711 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [6,388 kB]
Get:13 https://ppa.launchpadcontent.net/deadsn

In [3]:
# =========================
# CELL 2A — Install Dependencies
# =========================
!pip -q install chromadb rank-bm25 sentence-transformers rapidfuzz


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 99.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 109.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 112.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 113.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.

In [4]:
# =========================
# CELL 3 — Set Paths (FYP Root + One Shared ChromaDB)
# =========================
import os

# ---- BOOK IDENTIFIERS (keep SAME metadata keys as Science notebook) ----
GRADE   = 6
SUBJECT = "computer"         # keep lowercase for consistency
BOOK    = "CS6"              # short book code
DOC_ID  = "CS6"              # used for filtering + unique chunk ids
UNIT    = ""                 # optional global; we store unit info in 'chapter' via headings

# ---- FOLDERS (match your Science notebook structure) ----
FYP_ROOT = "/content/drive/MyDrive/FYP"

BOOK_DIR = f"{FYP_ROOT}/Grade_{GRADE}/Comp"     # change if your folder is different
OUT_DIR  = f"{BOOK_DIR}/outputs_cs6"             # per-book outputs (jsonl, caches, etc.)

# One shared Chroma DB for ALL grades/subjects/books
shared_db_dir     = f"{FYP_ROOT}/OneSharedChromaDB"
shared_collection = "ptb_textbooks"

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(shared_db_dir, exist_ok=True)

# ---- INPUT MARKDOWN ----
# Put your MD in BOOK_DIR, OR set the direct path here.
# Example if you uploaded to Colab (left panel): "/content/comp_6.md.md"
INPUT_MD = None  # keep None to auto-find the newest .md/.md.md inside BOOK_DIR

print("BOOK_DIR:", BOOK_DIR)
print("OUT_DIR :", OUT_DIR)
print("shared_db_dir:", shared_db_dir)


BOOK_DIR: /content/drive/MyDrive/FYP/Grade_6/Comp
OUT_DIR : /content/drive/MyDrive/FYP/Grade_6/Comp/outputs_cs6
shared_db_dir: /content/drive/MyDrive/FYP/OneSharedChromaDB


In [5]:
# =========================
# CELL 4 — Reset Book Outputs (SAFE)
# =========================
import shutil

blocks_dir = f"{OUT_DIR}/blocks"
cache_dir  = f"{OUT_DIR}/cache"

for p in [blocks_dir, cache_dir]:
    if os.path.exists(p):
        shutil.rmtree(p)
    os.makedirs(p, exist_ok=True)

print("Reset done:", blocks_dir, cache_dir)

# ---- OPTIONAL (DANGER): delete ONLY this book from shared ChromaDB ----
# (Uncomment ONLY if you want to re-ingest CS6 from scratch.)
# import chromadb
# client = chromadb.PersistentClient(path=shared_db_dir)
# col = client.get_or_create_collection(name=shared_collection)
# col.delete(where={"doc_id": DOC_ID})
# print("Deleted from Chroma where doc_id =", DOC_ID)


Reset done: /content/drive/MyDrive/FYP/Grade_6/Comp/outputs_cs6/blocks /content/drive/MyDrive/FYP/Grade_6/Comp/outputs_cs6/cache


In [6]:
# =========================
# CELL 5 — Locate Computer Markdown (Auto)
# =========================
import glob, os

def pick_input_md(book_dir: str):
    cands = []
    cands += glob.glob(os.path.join(book_dir, "*.md"))
    if not cands:
        return None
    # choose newest
    cands.sort(key=lambda p: os.path.getmtime(p), reverse=True)
    return cands[0]

if INPUT_MD is None:
    INPUT_MD = pick_input_md(BOOK_DIR)

if INPUT_MD is None or (not os.path.exists(INPUT_MD)):
    raise FileNotFoundError(
        "No markdown found. Put your MD in BOOK_DIR or set INPUT_MD manually.\n"
        f"BOOK_DIR={BOOK_DIR}"
    )

print("INPUT_MD:", INPUT_MD)


INPUT_MD: /content/drive/MyDrive/FYP/Grade_6/Comp/comp_6.md


In [7]:
# =========================
# CELL 6 — Markdown Cleaning Utilities (Tables, Noise, Units)
# =========================
import re
from html import unescape

def normalize_unit_h1(md_text: str) -> str:
    """
    Computer MD often has:
      # UNIT 2
      # Components of Computer System
    We merge into:
      # UNIT 2 — Components of Computer System
    """
    lines = md_text.splitlines()
    out = []
    i = 0
    h1_re = re.compile(r"^\s*#\s+(.*)\s*$")
    unit_re = re.compile(r"^\s*UNIT\s+\d+\s*$", re.IGNORECASE)

    while i < len(lines):
        m1 = h1_re.match(lines[i])
        if m1:
            t1 = m1.group(1).strip()
            # if this h1 is UNIT and next line is another h1 title, merge
            if unit_re.match(t1) and i + 1 < len(lines):
                m2 = h1_re.match(lines[i + 1])
                if m2:
                    t2 = m2.group(1).strip()
                    out.append(f"# {t1} — {t2}")
                    i += 2
                    continue
        out.append(lines[i])
        i += 1
    return "\n".join(out)

def html_table_to_text(md_text: str) -> str:
    """
    Convert <table>...</table> blocks into plain text rows to keep content searchable.
    """
    def repl_table(match):
        table_html = match.group(0)
        # rows
        rows = re.findall(r"<tr.*?>(.*?)</tr>", table_html, flags=re.DOTALL|re.IGNORECASE)
        out_rows = []
        for r in rows:
            cells = re.findall(r"<t[dh].*?>(.*?)</t[dh]>", r, flags=re.DOTALL|re.IGNORECASE)
            cells = [re.sub(r"<.*?>", "", c) for c in cells]
            cells = [unescape(re.sub(r"\s+", " ", c).strip()) for c in cells if c.strip()]
            if cells:
                out_rows.append(" | ".join(cells))
        if out_rows:
            return "\n" + "\n".join(out_rows) + "\n"
        return "\n"
    return re.sub(r"<table.*?>.*?</table>", repl_table, md_text, flags=re.DOTALL|re.IGNORECASE)

def strip_noise(md_text: str) -> str:
    # remove some repeated UI noise lines if present
    md_text = re.sub(r"(?m)^\s*Visible:\s*\d+%\s*-\s*\d+%\s*$", "", md_text)
    return md_text

def clean_md(md_text: str) -> str:
    md_text = normalize_unit_h1(md_text)
    md_text = strip_noise(md_text)
    md_text = html_table_to_text(md_text)
    md_text = unescape(md_text)

    # keep image descriptions but normalize
    md_text = re.sub(r"\[(The image shows.*?)\]", r"IMAGE: \1", md_text, flags=re.IGNORECASE)

    # normalize whitespace
    md_text = re.sub(r"\r\n", "\n", md_text)
    md_text = re.sub(r"\n{3,}", "\n\n", md_text)
    return md_text.strip()

raw_md = open(INPUT_MD, "r", encoding="utf-8", errors="ignore").read()
md_clean = clean_md(raw_md)

print("Raw chars:", len(raw_md))
print("Clean chars:", len(md_clean))
print(md_clean[:600])


Raw chars: 211449
Clean chars: 196817
# COMPUTER SCIENCE 6
Based on Single National Curriculum 2022

The cover features several illustrations related to technology and digital learning:
- A large number "6" in a blue circle with a white border.
- An image of a human profile silhouette filled with various digital icons and glowing circuits, representing artificial intelligence or digital thought.
- An image of a laptop screen displaying the text "E LEARNING" surrounded by educational icons like a graduation cap, a book, and a pencil.
- A central large illustration showing a global network connecting various devices including a desk


In [8]:
# =========================
# CELL 7 — Parse Markdown into Heading Blocks (h1/h2/h3 + text)
# =========================
import re

HEADING_RE = re.compile(r"^(#{1,3})\s+(.*)\s*$")

def md_to_blocks(md_text: str):
    blocks = []
    h1 = h2 = h3 = ""
    buff = []

    def flush():
        nonlocal buff
        text = "\n".join(buff).strip()
        if text:
            blocks.append({"h1": h1, "h2": h2, "h3": h3, "text": text})
        buff = []

    for line in md_text.splitlines():
        m = HEADING_RE.match(line)
        if m:
            flush()
            lvl = len(m.group(1))
            title = m.group(2).strip()
            if lvl == 1:
                h1 = title
                h2 = ""
                h3 = ""
            elif lvl == 2:
                h2 = title
                h3 = ""
            else:
                h3 = title
            continue
        buff.append(line)

    flush()
    return blocks

blocks = md_to_blocks(md_clean)
print("Blocks:", len(blocks))
print("Sample block keys:", blocks[0].keys())
print("\nH1:", blocks[0]["h1"])
print("H2:", blocks[0]["h2"])
print("H3:", blocks[0]["h3"])
print(blocks[0]["text"][:400])


Blocks: 273
Sample block keys: dict_keys(['h1', 'h2', 'h3', 'text'])

H1: COMPUTER SCIENCE 6
H2: 
H3: 
Based on Single National Curriculum 2022

The cover features several illustrations related to technology and digital learning:
- A large number "6" in a blue circle with a white border.
- An image of a human profile silhouette filled with various digital icons and glowing circuits, representing artificial intelligence or digital thought.
- An image of a laptop screen displaying the text "E LEARNIN


In [9]:
# =========================
# CELL 8 — Detect Block Type (SLO / KEY_POINTS / EXERCISE / etc.)
# =========================
import re

def detect_block_type(h1: str, h2: str, h3: str, text: str) -> str:
    title = " ".join([h1 or "", h2 or "", h3 or ""]).lower()
    t0 = (text[:600] if text else "").lower()

    if "students learning outcomes" in title or "learning outcomes" in title:
        return "SLO"

    # exercises / questions / activities
    if any(k in title for k in ["exercise", "questions", "tick", "fill", "project based", "activity based"]):
        return "EXERCISE"
    if any(k in t0 for k in ["answer the following", "answer the questions", "fill in the blanks", "tick", "briefly"]):
        return "EXERCISE"

    # glossary / weblinks
    if "glossary" in title:
        return "GLOSSARY"
    if any(k in title for k in ["weblinks", "web links", "links"]):
        return "WEBLINKS"

    # key points / notes / extra bit / do you know / summary
    if any(k in title for k in ["summary", "key points"]):
        return "KEY_POINTS"
    if any(k in t0 for k in ["**do you know", "**extra bit", "**note:"]):
        return "KEY_POINTS"

    return "BODY"

print(detect_block_type("UNIT 1 — ICT Fundamentals", "Students Learning Outcomes", "", blocks[1]["text"]))


SLO


In [10]:
# =========================
# CELL 9 — Build Typed Block Objects + Save JSONL
# =========================
import json, os

typed_blocks = []
for b in blocks:
    chapter = (b["h1"] or "").strip()
    section = (b["h2"] or b["h3"] or "").strip()
    block_type = detect_block_type(b["h1"], b["h2"], b["h3"], b["text"])

    typed_blocks.append({
        "doc_id": DOC_ID,
        "grade": GRADE,
        "subject": SUBJECT,
        "book": BOOK,
        "unit": UNIT,                 # keep key for consistency
        "chapter": chapter,           # holds UNIT info (after merge)
        "section": section,
        "block_type": block_type,
        "source_type": "markdown_clean",
        "source_pdf": "",             # keep key (fill if you want)
        "text": b["text"].strip()
    })

blocks_jsonl = os.path.join(blocks_dir, f"{DOC_ID}_typed_blocks.jsonl")
with open(blocks_jsonl, "w", encoding="utf-8") as f:
    for x in typed_blocks:
        f.write(json.dumps(x, ensure_ascii=False) + "\n")

print("typed_blocks:", len(typed_blocks))
print("Saved:", blocks_jsonl)
print("Example:", typed_blocks[0])


typed_blocks: 273
Saved: /content/drive/MyDrive/FYP/Grade_6/Comp/outputs_cs6/blocks/CS6_typed_blocks.jsonl
Example: {'doc_id': 'CS6', 'grade': 6, 'subject': 'computer', 'book': 'CS6', 'unit': '', 'chapter': 'COMPUTER SCIENCE 6', 'section': '', 'block_type': 'BODY', 'source_type': 'markdown_clean', 'source_pdf': '', 'text': 'Based on Single National Curriculum 2022\n\nThe cover features several illustrations related to technology and digital learning:\n- A large number "6" in a blue circle with a white border.\n- An image of a human profile silhouette filled with various digital icons and glowing circuits, representing artificial intelligence or digital thought.\n- An image of a laptop screen displaying the text "E LEARNING" surrounded by educational icons like a graduation cap, a book, and a pencil.\n- A central large illustration showing a global network connecting various devices including a desktop computer, a laptop, a tablet, and a smartphone to a globe, set against a background o

In [11]:
# =========================
# CELL 10 — Chunking (Sentence + Paragraph, keeps code blocks)
# =========================
import re

SENT_SPLIT = re.compile(r"(?<=[.!?])\s+")

def split_keep_codeblocks(text: str):
    """
    Split into segments where ```...``` blocks stay intact.
    """
    parts = []
    code_re = re.compile(r"```.*?```", flags=re.DOTALL)
    last = 0
    for m in code_re.finditer(text):
        if m.start() > last:
            parts.append(("text", text[last:m.start()]))
        parts.append(("code", m.group(0)))
        last = m.end()
    if last < len(text):
        parts.append(("text", text[last:]))
    return parts

def chunk_text(text: str, max_chars=1200, overlap=180):
    text = (text or "").strip()
    if not text:
        return []

    parts = split_keep_codeblocks(text)
    units = []
    for kind, part in parts:
        part = part.strip()
        if not part:
            continue
        if kind == "code":
            units.append(part)
        else:
            # split by paragraphs first
            paras = [p.strip() for p in re.split(r"\n{2,}", part) if p.strip()]
            for p in paras:
                units.append(p)

    chunks = []
    cur = ""
    for u in units:
        if len(cur) + len(u) + 2 <= max_chars:
            cur = (cur + "\n\n" + u).strip() if cur else u
        else:
            if cur:
                chunks.append(cur.strip())
            # if u itself too large, split by sentences
            if len(u) > max_chars:
                sents = [s.strip() for s in SENT_SPLIT.split(u) if s.strip()]
                tmp = ""
                for s in sents:
                    if len(tmp) + len(s) + 1 <= max_chars:
                        tmp = (tmp + " " + s).strip() if tmp else s
                    else:
                        if tmp:
                            chunks.append(tmp)
                        tmp = s
                if tmp:
                    chunks.append(tmp)
                cur = ""
            else:
                cur = u

    if cur:
        chunks.append(cur.strip())

    # add overlap (soft overlap by tail chars)
    if overlap > 0 and len(chunks) > 1:
        out = [chunks[0]]
        for i in range(1, len(chunks)):
            prev = out[-1]
            tail = prev[-overlap:] if len(prev) > overlap else prev
            out.append((tail + "\n" + chunks[i]).strip())
        chunks = out

    return chunks

# quick sanity
test_chunks = chunk_text(typed_blocks[0]["text"], max_chars=800)
print("chunks:", len(test_chunks))
print(test_chunks[0][:400])


chunks: 1
Based on Single National Curriculum 2022

The cover features several illustrations related to technology and digital learning:
- A large number "6" in a blue circle with a white border.
- An image of a human profile silhouette filled with various digital icons and glowing circuits, representing artificial intelligence or digital thought.
- An image of a laptop screen displaying the text "E LEARNIN


In [12]:
# =========================
# CELL 11 — Convert Typed Blocks → Chunks (same metadata keys)
# =========================
import hashlib

def sha1_text(s: str) -> str:
    return hashlib.sha1(s.encode("utf-8", errors="ignore")).hexdigest()

chunks = []
for b in typed_blocks:
    pieces = chunk_text(b["text"], max_chars=1200, overlap=180)
    for p in pieces:
        h = sha1_text(p)
        chunk_id = f"{DOC_ID}__{len(chunks):06d}__{h[:8]}"
        chunks.append({
            "chunk_id": chunk_id,
            "doc_id": b["doc_id"],
            "grade": b["grade"],
            "subject": b["subject"],
            "book": b["book"],
            "unit": b["unit"],
            "chapter": b["chapter"],
            "section": b["section"],
            "block_type": b["block_type"],
            "source_type": b["source_type"],
            "source_pdf": b["source_pdf"],
            "text": p,
            "text_hash": h
        })

chunks_jsonl = os.path.join(blocks_dir, f"{DOC_ID}_chunks.jsonl")
with open(chunks_jsonl, "w", encoding="utf-8") as f:
    for x in chunks:
        f.write(json.dumps(x, ensure_ascii=False) + "\n")

print("Total chunks:", len(chunks))
print("Saved:", chunks_jsonl)
print("Example:", chunks[0].keys())


Total chunks: 319
Saved: /content/drive/MyDrive/FYP/Grade_6/Comp/outputs_cs6/blocks/CS6_chunks.jsonl
Example: dict_keys(['chunk_id', 'doc_id', 'grade', 'subject', 'book', 'unit', 'chapter', 'section', 'block_type', 'source_type', 'source_pdf', 'text', 'text_hash'])


In [13]:
# =========================
# CELL 12 — Load Embedder + Build Embeddings (batched)
# =========================
from sentence_transformers import SentenceTransformer
import numpy as np
import torch

EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
device = "cuda" if torch.cuda.is_available() else "cpu"
embedder = SentenceTransformer(EMBED_MODEL_NAME, device=device)

BATCH = 64

docs  = [c["text"] for c in chunks]
ids   = [c["chunk_id"] for c in chunks]

metas = []
for c in chunks:
    m = {k: c[k] for k in [
        "chunk_id","doc_id","grade","subject","book","unit","chapter","section",
        "block_type","source_type","source_pdf","text_hash"
    ]}
    m["embed_model"] = EMBED_MODEL_NAME
    metas.append(m)

embeddings = []
for i in range(0, len(docs), BATCH):
    batch_docs = docs[i:i+BATCH]
    emb = embedder.encode(batch_docs, normalize_embeddings=True, show_progress_bar=False)
    embeddings.append(emb)

embeddings = np.vstack(embeddings).astype("float32")
print("Embeddings shape:", embeddings.shape, "device:", device)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings shape: (319, 384) device: cuda


In [14]:
# =========================
# CELL 13 — Connect to Shared ChromaDB (OneSharedChromaDB)
# =========================
import chromadb

client = chromadb.PersistentClient(path=shared_db_dir)
collection = client.get_or_create_collection(name=shared_collection)

print("Collection:", shared_collection)
print("Chroma path:", shared_db_dir)


Collection: ptb_textbooks
Chroma path: /content/drive/MyDrive/FYP/OneSharedChromaDB


In [15]:
# =========================
# CELL 14 — Upsert into Shared ChromaDB (batched)
# =========================
BATCH_UPSERT = 2000

for i in range(0, len(ids), BATCH_UPSERT):
    j = i + BATCH_UPSERT
    collection.upsert(
        ids=ids[i:j],
        documents=docs[i:j],
        metadatas=metas[i:j],
        embeddings=embeddings[i:j].tolist()
    )
    print(f"Upserted {i}..{j}")

print("Done. Total upserted:", len(ids))


Upserted 0..2000
Done. Total upserted: 319


Retriever

In [48]:
# =========================
# CELL 15R — Build BM25 Index (lexical) from loaded docs
# =========================
from rank_bm25 import BM25Okapi
import re
import numpy as np

STOP = set("""
a an the and or but if then else is are was were to of in on for from with as by at into
this that these those it its be been being we you they he she i
""".split())

# extra "question words" that hurt matching
QSTOP = set("""
define explain tell write what why how steps procedure give examples example simple words name list
differentiate difference between
""".split())

def tokenize(s: str):
    s = (s or "").lower()
    toks = re.findall(r"[a-z0-9_]+", s)
    return [t for t in toks if t not in STOP and t not in QSTOP and len(t) > 1]

# ✅ IMPORTANT: include headings in BM25 text (helps a LOT)
bm25_docs = []
for d, m in zip(docs, metas):
    ch = (m.get("chapter") or "")
    sec = (m.get("section") or "")
    bm25_docs.append(f"{ch}\n{sec}\n{d}")

bm25_corpus = [tokenize(d) for d in bm25_docs]
bm25 = BM25Okapi(bm25_corpus)

print("✅ BM25 ready. Docs:", len(bm25_docs))


✅ BM25 ready. Docs: 319


In [77]:
# =========================
# CELL 16 — Intent + Block-Type Allowlist (fast) [UPDATED]
# =========================
import re

def detect_intent(q: str) -> str:
    ql = (q or "").strip().lower()

    # EXERCISE intent
    if any(k in ql for k in ["exercise", "mcq", "fill in", "fill-in", "tick", "briefly", "question", "true/false"]):
        return "EXERCISE"

    # PROCEDURE intent (how-to / steps)
    if any(k in ql for k in ["steps", "how to", "procedure", "create", "open", "click", "right click", "right-click", "shortcut", "delete", "copy", "paste"]):
        return "PROCEDURE"

    # DEFINE intent (definitions + "what are" + "give examples")
    if re.search(r"\b(define|definition|meaning of|what is|what are|stands for|differentiate|difference between)\b", ql):
        return "DEFINE"

    # CODE intent ONLY if user clearly asks for code/program
    if any(k in ql for k in ["write a program", "program in", "code", "pseudocode", "algorithm", "scratch code", "python code"]):
        return "CODE"

    return "GENERAL"

def allowed_block_types(intent: str):
    if intent in ["PROCEDURE", "CODE", "GENERAL", "DEFINE"]:
        return {"BODY", "KEY_POINTS", "GLOSSARY"}
    if intent == "EXERCISE":
        return {"EXERCISE", "BODY", "KEY_POINTS"}
    return {"BODY", "KEY_POINTS"}

BLOCK_WEIGHT = {
    "BODY": 1.00,
    "KEY_POINTS": 0.90,
    "GLOSSARY": 0.95,
    "EXERCISE": 0.80,
    "SLO": 0.45,
    "WEBLINKS": 0.10
}
print("✅ Intent + allowlist ready (UPDATED)")


✅ Intent + allowlist ready (UPDATED)


In [18]:
# =========================
# CELL 17 — Dense Retrieve from Chroma (metadata filter FIRST)
# =========================
def chroma_dense(query: str, top_k=25):
    q_emb = embedder.encode([query], normalize_embeddings=True)[0].astype("float32").tolist()

    res = collection.query(
        query_embeddings=[q_emb],
        n_results=top_k,
        where={"doc_id": DOC_ID},
        include=["documents", "metadatas", "distances"]
    )

    out = []
    docs_ = res["documents"][0]
    metas_= res["metadatas"][0]
    dists_= res["distances"][0]  # smaller = closer
    for doc, meta, dist in zip(docs_, metas_, dists_):
        out.append({"text": doc, "meta": meta, "score": float(1.0/(1.0+dist)), "src": "dense"})
    return out
print("✅ chroma_dense ready")


✅ chroma_dense ready


In [50]:
# =========================
# CELL 18 — BM25 Retrieve (lexical)
# =========================
def bm25_lexical(query: str, top_k=25):
    q_toks = tokenize(query)
    scores = bm25.get_scores(q_toks)
    idxs = np.argsort(scores)[::-1][:top_k]
    out = []
    for idx in idxs:
        out.append({"text": docs[idx], "meta": metas[idx], "score": float(scores[idx]), "src": "bm25"})
    return out

print("✅ bm25_lexical ready")


✅ bm25_lexical ready


In [76]:
# =========================
# CELL 19 — Hybrid Fusion (RRF) + Type Filter + Stronger Gate [UPDATED]
# =========================
import re

def keyword_gate(query: str, passage: str, intent: str = "GENERAL") -> bool:
    """
    Stricter gate:
    - For DEFINE/GENERAL: require stronger token overlap
    - For PROCEDURE: allow slightly looser (steps may use varied wording)
    """
    q_toks = tokenize(query)
    if not q_toks:
        return True

    p_toks = set(tokenize((passage or "")[:1400]))
    overlap = len(set(q_toks) & p_toks)

    # Short queries like "Define ICT" should pass with 1 overlap
    if len(q_toks) <= 2:
        return overlap >= 1

    if intent in ["PROCEDURE"]:
        return overlap >= 1
    else:
        # DEFINE / GENERAL / CODE: be stricter
        return overlap >= 2

def rrf_fuse(list_a, list_b, k=60):
    pool = {}
    def add(lst):
        for r, item in enumerate(lst, start=1):
            cid = (item["meta"] or {}).get("chunk_id") or (item["meta"] or {}).get("text_hash") or f"rank_{r}"
            if cid not in pool:
                pool[cid] = {**item, "rrf": 0.0}
            pool[cid]["rrf"] += 1.0 / (k + r)

    add(list_a)
    add(list_b)

    # block-type weighting
    for cid, it in pool.items():
        bt = ((it.get("meta") or {}).get("block_type") or "BODY").strip().upper()
        it["rrf"] *= BLOCK_WEIGHT.get(bt, 0.8)

    return sorted(pool.values(), key=lambda x: x["rrf"], reverse=True)

def retrieve_hybrid(query: str, top_k=8, dense_k=40, bm25_k=40):
    intent = detect_intent(query)
    allowed = allowed_block_types(intent)

    dense = chroma_dense(query, top_k=dense_k)
    lex   = bm25_lexical(query, top_k=bm25_k)

    fused = rrf_fuse(dense, lex, k=60)

    final = []
    seen = set()
    for it in fused:
        meta = it.get("meta") or {}
        cid = meta.get("chunk_id") or meta.get("text_hash")
        if cid in seen:
            continue
        seen.add(cid)

        bt = (meta.get("block_type") or "BODY").strip().upper()
        if bt not in allowed:
            continue
        if bt == "WEBLINKS":
            continue
        if not keyword_gate(query, it.get("text",""), intent=intent):
            continue

        final.append(it)
        if len(final) >= top_k:
            break

    return {"intent": intent, "hits": final}

# quick test
test = retrieve_hybrid("What are input devices? Give 4 examples.", top_k=5)
print("Intent:", test["intent"])
for h in test["hits"]:
    m = h["meta"] or {}
    print("-", m.get("chapter",""), "|", m.get("section",""), "|", m.get("block_type",""), "|", m.get("chunk_id",""))


Intent: GENERAL
- UNIT 1 — ICT Fundamentals | 1.3 Software | BODY | CS6__000015__c0f65c90
- UNIT 2 — Components of Computer System | 2.6 Working of Computer System | BODY | CS6__000076__145ffc21
- UNIT 1 — ICT Fundamentals | 1.2 Computer | BODY | CS6__000008__01a78054
- UNIT 2 — Components of Computer System | 2.4 Output Devices | BODY | CS6__000051__9aca4c95
- UNIT 1 — ICT Fundamentals | 1.6 ICT Devices | BODY | CS6__000022__a214acc5


In [52]:
# =========================
# CELL 20 — Build RAG Prompt (book-only)
# =========================
def build_prompt(question: str, hits: list, max_chars_ctx=4500):
    ctx_parts = []
    used = 0
    for i, h in enumerate(hits, start=1):
        cid = h["meta"].get("chunk_id","")
        ch  = h["meta"].get("chapter","")
        sec = h["meta"].get("section","")
        piece = f"[{i}] (chunk_id={cid}) (chapter={ch}) (section={sec})\n{h['text'].strip()}\n"
        if used + len(piece) > max_chars_ctx:
            break
        ctx_parts.append(piece)
        used += len(piece)

    context = "\n---\n".join(ctx_parts)

    prompt = f"""You are a helpful tutor for Grade {GRADE} {SUBJECT.title()}.
RULES:
- Answer ONLY from the provided context.
- If the context is insufficient, say: "I don't have that information in the book context."
- Use simple words for a child.
- Prefer step-by-step when the question is procedural.
- End with: Sources: [chunk_id1, chunk_id2, ...]

QUESTION:
{question}

CONTEXT:
{context}

ANSWER:
"""
    return prompt

# demo prompt build
demo = retrieve_hybrid("Tell me steps to create a new folder in Windows.", top_k=5)
prompt = build_prompt("Tell me steps to create a new folder in Windows.", demo["hits"])
print(prompt[:900])


You are a helpful tutor for Grade 6 Computer.
RULES:
- Answer ONLY from the provided context.
- If the context is insufficient, say: "I don't have that information in the book context."
- Use simple words for a child.
- Prefer step-by-step when the question is procedural.
- End with: Sources: [chunk_id1, chunk_id2, ...]

QUESTION:
Tell me steps to create a new folder in Windows.

CONTEXT:
[1] (chunk_id=CS6__000114__b7b31ae2) (chapter=Types of Operating Systems) (section=3.6 Stepping into Windows)
To create a folder is almost the same as creating a file. The following are the simple steps to create a new folder:

**Step 1:**
In the location where you want to create a folder, select the option of the folder to create the folder.

IMAGE: The image shows a Windows File Explorer window with a right-click context menu open. The user has navigated to 'New' and a red arrow points to the 'Folder'


In [53]:
# =========================
# CELL 21 — Install + Import LLM Tools (Transformers + BitsAndBytes)
# =========================
!pip -q install transformers accelerate bitsandbytes huggingface_hub
print("✅ Installed transformers stack")


✅ Installed transformers stack


In [23]:
# =========================
# CELL 22 — Hugging Face Login (Only if using gated Llama)
# =========================
import os, getpass
from huggingface_hub import login

# If you already set token earlier, this will just reuse it.
if "HF_TOKEN" not in os.environ or not os.environ["HF_TOKEN"].strip():
    os.environ["HF_TOKEN"] = getpass.getpass("Enter your HuggingFace token (read access): ")

login(token=os.environ["HF_TOKEN"])
print("✅ Hugging Face login done")


Enter your HuggingFace token (read access): ··········


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


✅ Hugging Face login done


In [24]:
# =========================
# CELL 23 — Load Llama 3.1 8B Instruct (4-bit if GPU; fallback if CPU)
# =========================
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Primary (gated) model
LLM_MODEL_ID = "meta-llama/Meta-Llama-3.1-8B-Instruct"

# Safe fallback (not gated) if GPU missing / loading fails
FALLBACK_MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

use_gpu = torch.cuda.is_available()
print("GPU available:", use_gpu)

def _load_model(model_id: str):
    if use_gpu:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.bfloat16
        )
        load_kwargs = dict(
            device_map="auto",
            quantization_config=bnb_config,
            torch_dtype=torch.bfloat16
        )
    else:
        # CPU: do NOT attempt 8B; fallback will load here
        load_kwargs = dict(
            device_map="cpu",
            torch_dtype=torch.float32
        )

    tok = AutoTokenizer.from_pretrained(model_id, use_fast=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    mdl = AutoModelForCausalLM.from_pretrained(model_id, **load_kwargs)
    mdl.eval()
    return tok, mdl

try:
    if not use_gpu:
        raise RuntimeError("No GPU: using fallback on CPU to avoid RAM crash.")

    tokenizer, model = _load_model(LLM_MODEL_ID)
    ACTIVE_MODEL_ID = LLM_MODEL_ID
    print("✅ Loaded:", ACTIVE_MODEL_ID)

except Exception as e:
    print("⚠️ Llama load failed. Using fallback model.")
    print("Reason:", str(e)[:300])
    tokenizer, model = _load_model(FALLBACK_MODEL_ID)
    ACTIVE_MODEL_ID = FALLBACK_MODEL_ID
    print("✅ Loaded:", ACTIVE_MODEL_ID)


GPU available: True


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

✅ Loaded: meta-llama/Meta-Llama-3.1-8B-Instruct


In [81]:
# =========================
# CELL 24 — LLM Generate Helper (NO PROMPT ECHO) [UPDATED]
# =========================
import torch

def format_chat_prompt(user_prompt: str) -> str:
    if hasattr(tokenizer, "apply_chat_template"):
        messages = [
            {"role": "system", "content": "You are a helpful tutor. Follow the rules strictly."},
            {"role": "user", "content": user_prompt}
        ]
        try:
            return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        except Exception:
            pass
    return user_prompt

@torch.inference_mode()
def llm_generate(prompt: str, max_new_tokens=220, temperature=0.2, top_p=0.9):
    text_in = format_chat_prompt(prompt)

    inputs = tokenizer(text_in, return_tensors="pt", padding=True)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    gen = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=(temperature > 0),
        temperature=temperature,
        top_p=top_p,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

    # ✅ decode ONLY newly generated tokens (prevents prompt echo)
    in_len = inputs["input_ids"].shape[1]
    new_tokens = gen[0][in_len:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

print("✅ llm_generate ready (UPDATED) | model:", ACTIVE_MODEL_ID)


✅ llm_generate ready (UPDATED) | model: meta-llama/Meta-Llama-3.1-8B-Instruct


In [82]:
# =========================
# CELL 25 — RAG Answer (Easy Words + Textbook-Only + Evidence IDs)
# =========================
import re

def select_context_hits(hits, max_chars=4500):
    """
    Prefer BODY/KEY_POINTS/GLOSSARY; avoid SLO/WEBLINKS unless nothing else exists.
    Returns: (context_str, used_chunk_ids)
    """
    prefer = []
    backup = []
    for h in hits:
        bt = (h.get("meta") or {}).get("block_type", "BODY")
        if bt in ["SLO", "WEBLINKS"]:
            backup.append(h)
        else:
            prefer.append(h)

    ordered = prefer + backup

    ctx = []
    used = 0
    used_ids = []

    for i, h in enumerate(ordered, start=1):
        meta = h.get("meta") or {}
        cid = meta.get("chunk_id", "")
        ch  = meta.get("chapter", "")
        sec = meta.get("section", "")
        bt  = meta.get("block_type", "")

        piece = f"[{i}] chunk_id={cid} | {bt} | {ch} | {sec}\n{h['text'].strip()}\n"
        if used + len(piece) > max_chars:
            break

        ctx.append(piece)
        used += len(piece)
        if cid:
            used_ids.append(cid)

    return "\n---\n".join(ctx), used_ids

def rag_prompt(question: str, context: str) -> str:
    return f"""You are a Grade {GRADE} Computer Science tutor.

RULES (VERY IMPORTANT):
- Use ONLY the provided CONTEXT (textbook chunks).
- If the context is not enough, say exactly: "I don't have that information in the book context."
- Explain in simple words.
- If the question asks steps, answer in numbered steps.
- End with: Sources: [chunk_id1, chunk_id2, ...]

QUESTION:
{question}

CONTEXT:
{context}

ANSWER:
"""

def _finalize_sources(answer_text: str, chunk_ids: list):
    # remove ANY existing Sources lines the model may produce
    answer_text = re.sub(r"(?im)^\s*sources\s*:\s*\[.*?\]\s*$", "", answer_text).strip()

    # unique, keep order
    seen = set()
    uniq = []
    for c in chunk_ids:
        if c and c not in seen:
            uniq.append(c)
            seen.add(c)

    return answer_text + "\n\nSources: [" + ", ".join(uniq) + "]"

def rag_answer(question: str, top_k=6):
    pack = retrieve_hybrid(question, top_k=top_k)
    hits = pack["hits"]

    context, used_chunk_ids = select_context_hits(hits, max_chars=4500)
    prompt = rag_prompt(question, context)

    answer = llm_generate(prompt, max_new_tokens=260, temperature=0.2)

    answer = _finalize_sources(answer, used_chunk_ids[:top_k])

    return {
        "intent": pack.get("intent"),
        "answer": answer,
        "sources": used_chunk_ids[:top_k],
        "hits": hits
    }

print("✅ rag_answer upgraded (clean chunk_id sources)")


✅ rag_answer upgraded (clean chunk_id sources)


In [83]:
# =========================
# CELL 26 — Test Full RAG Answer
# =========================
q1 = "Define ICT in simple words."
r1 = rag_answer(q1, top_k=6)
print("Q:", q1)
print(r1["answer"])
print("Sources:", r1["sources"])

print("\n" + "="*90 + "\n")

q2 = "What are input devices? Give 3 examples."
r2 = rag_answer(q2, top_k=6)
print("Q:", q2)
print(r2["answer"])
print("Sources:", r2["sources"])


Q: Define ICT in simple words.
ICT stands for Information and Communication Technology. It is a set of computing tools that helps people and organizations interact in the digital world. ICT includes devices like computers, phones, and networks that allow us to communicate, access information, and perform various tasks.

Sources: [CS6__000023__418b7cf6, CS6__000027__af3505c2, CS6__000007__7d79d3d7, CS6__000028__d1a3c74c, CS6__000029__acddffa4, CS6__000022__a214acc5]
Sources: ['CS6__000023__418b7cf6', 'CS6__000027__af3505c2', 'CS6__000007__7d79d3d7', 'CS6__000028__d1a3c74c', 'CS6__000029__acddffa4', 'CS6__000022__a214acc5']


Q: What are input devices? Give 3 examples.
Input devices are hardware used to give data to the computer. Here are 3 examples of input devices:

1. Keyboard: It is used to type commands and data into the computer.
2. Mouse: It is used to interact with the computer by clicking on icons and buttons.
3. Microphone: It is used to input audio data into the computer.

Sou

Retriever Testing + Metrics

In [84]:
# =========================
# CELL 27 — Create Evaluation Query Set (CS6)
# =========================
import os, json
from datetime import datetime

EVAL_DIR = os.path.join(OUT_DIR, "eval")
os.makedirs(EVAL_DIR, exist_ok=True)

EVAL_QUERIES = [
    "Define ICT in simple words.",
    "Write advantages of ICT in daily life.",
    "Explain OCR and what it does.",
    "What is a computer system? Name its components.",
    "What are input devices? Give 4 examples.",
    "What are output devices? Give 4 examples.",
    "Explain CPU in simple words. What does CPU do?",
    "How to create a new folder in Windows? Write steps.",
    "What is Scratch? Why is Scratch used for learning?",
    "Differentiate between hardware and software."
]

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
EVAL_RUN_PATH    = os.path.join(EVAL_DIR, f"{DOC_ID}_eval_run_{RUN_ID}.jsonl")
EVAL_LABELS_PATH = os.path.join(EVAL_DIR, f"{DOC_ID}_eval_labels.jsonl")

print("✅ Eval queries:", len(EVAL_QUERIES))
print("Run file:", EVAL_RUN_PATH)
print("Labels file:", EVAL_LABELS_PATH)


✅ Eval queries: 10
Run file: /content/drive/MyDrive/FYP/Grade_6/Comp/outputs_cs6/eval/CS6_eval_run_20260207_083652.jsonl
Labels file: /content/drive/MyDrive/FYP/Grade_6/Comp/outputs_cs6/eval/CS6_eval_labels.jsonl


In [86]:
# =========================
# CELL 28 — Run Retriever on Eval Queries (Save Top-N hits)
# =========================
TOP_N = 10  # evaluate top-10 retrieval

def run_eval_retriever(queries, top_n=10):
    rows = []
    for q in queries:
        r = retrieve_hybrid(q, top_k=top_n)

        hits = []
        for h in r["hits"]:
            meta = h.get("meta") or {}
            hits.append({
                "chunk_id": meta.get("chunk_id", ""),
                "chapter": meta.get("chapter", ""),
                "section": meta.get("section", ""),
                "block_type": meta.get("block_type", ""),
                "src": h.get("src", ""),
                "score": float(h.get("rrf", h.get("score", 0.0)))
            })

        rows.append({
            "query": q,
            "intent": r.get("intent"),
            "top_n": top_n,
            "hits": hits
        })
    return rows

eval_rows = run_eval_retriever(EVAL_QUERIES, top_n=TOP_N)

with open(EVAL_RUN_PATH, "w", encoding="utf-8") as f:
    for r in eval_rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print("✅ Saved eval run:", EVAL_RUN_PATH)
print("Sample query:", eval_rows[0]["query"])
print("Top-1:", eval_rows[0]["hits"][0]["chunk_id"], "|", eval_rows[0]["hits"][0]["chapter"])



✅ Saved eval run: /content/drive/MyDrive/FYP/eval_runs/CS6_eval_run.jsonl
Sample query: Define ICT in simple words.
Top-1: CS6__000023__418b7cf6 | UNIT 1 — ICT Fundamentals


In [79]:
# =========================
# CELL 29 — Auto-Pick Latest Eval Run + Ensure Labels File
# =========================
import os, glob, json

# If you already have EVAL_DIR, DOC_ID, OUT_DIR from previous cells, this will work.
EVAL_DIR = os.path.join(OUT_DIR, "eval")
os.makedirs(EVAL_DIR, exist_ok=True)

def pick_latest_eval_run(eval_dir: str, doc_id: str):
    patt = os.path.join(eval_dir, f"{doc_id}_eval_run_*.jsonl")
    cands = glob.glob(patt)
    if not cands:
        return None
    cands.sort(key=lambda p: os.path.getmtime(p), reverse=True)
    return cands[0]

EVAL_RUN_PATH = pick_latest_eval_run(EVAL_DIR, DOC_ID)
if EVAL_RUN_PATH is None:
    raise FileNotFoundError("No eval run file found. Run CELL 28 first.")

EVAL_LABELS_PATH = os.path.join(EVAL_DIR, f"{DOC_ID}_eval_labels.jsonl")

print("✅ Using eval run:", EVAL_RUN_PATH)
print("✅ Labels file    :", EVAL_LABELS_PATH)

# Create labels file if not exists
if (not os.path.exists(EVAL_LABELS_PATH)) or os.path.getsize(EVAL_LABELS_PATH) < 5:
    # build empty labels based on eval run
    with open(EVAL_RUN_PATH, "r", encoding="utf-8") as f_in, open(EVAL_LABELS_PATH, "w", encoding="utf-8") as f_out:
        for line in f_in:
            row = json.loads(line)
            rel = {h["chunk_id"]: 0 for h in row.get("hits", []) if h.get("chunk_id")}
            f_out.write(json.dumps({"query": row["query"], "relevance": rel}, ensure_ascii=False) + "\n")
    print("✅ Created new labels template.")
else:
    print("✅ Labels file already exists (will update/append during labeling).")


✅ Using eval run: /content/drive/MyDrive/FYP/Grade_6/Comp/outputs_cs6/eval/CS6_eval_run_20260207_071453.jsonl
✅ Labels file    : /content/drive/MyDrive/FYP/Grade_6/Comp/outputs_cs6/eval/CS6_eval_labels.jsonl
✅ Labels file already exists (will update/append during labeling).


In [69]:
# # =========================
# # CELL 29 — Load / Save Labels
# # =========================
# import json, os

# EVAL_LABELS_PATH = os.path.join(EVAL_OUT_DIR, f"{DOC_ID}_labels.json")

# def load_labels(path):
#     if os.path.exists(path):
#         with open(path, "r", encoding="utf-8") as f:
#             return json.load(f)
#     return {}  # {query: {chunk_id: 1/2}}

# def save_labels(labels, path):
#     with open(path, "w", encoding="utf-8") as f:
#         json.dump(labels, f, ensure_ascii=False, indent=2)

# labels_db = load_labels(EVAL_LABELS_PATH)
# print("Loaded labels:", len(labels_db), "| path:", EVAL_LABELS_PATH)


NameError: name 'EVAL_OUT_DIR' is not defined

In [31]:
# =========================
# CELL 30 — Build chunk_id → text map (for accurate labeling)
# =========================
id_to_text = {m["chunk_id"]: d for d, m in zip(docs, metas)}
print("✅ id_to_text size:", len(id_to_text))


✅ id_to_text size: 319


In [32]:
# =========================
# CELL 31 — Preview hits with chunk TEXT (so you label correctly)
# =========================
import json

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(x) for x in f]

eval_rows = load_jsonl(EVAL_RUN_PATH)

def preview_query(query, k=10, chars=650):
    row = next((r for r in eval_rows if r["query"] == query), None)
    if row is None:
        print("❌ Query not found in eval run")
        return

    print("\n" + "="*110)
    print("QUERY:", row["query"])
    print("INTENT:", row.get("intent"))
    print("="*110)

    for i, h in enumerate(row["hits"][:k], start=1):
        cid = h.get("chunk_id","")
        ch  = h.get("chapter","")
        sec = h.get("section","")
        bt  = h.get("block_type","")

        txt = id_to_text.get(cid, "")
        snip = (txt[:chars] + "…") if len(txt) > chars else txt

        print(f"\n{i}. {cid} | {bt} | {ch} | {sec}")
        print("-"*90)
        print(snip)

# Example:
# preview_query("Define ICT in simple words.", k=10, chars=650)
print("✅ Use: preview_query(<your query string>)")


✅ Use: preview_query(<your query string>)


In [33]:
# =========================
# CELL 32 — Improved ready-to-copy auto-suggest (more accurate)
# =========================
import re, json

STOP = set("""
a an the and or but if then else is are was were to of in on for from with as by at into
this that these those it its be been being we you they he she i
""".split())

# extra query-words to ignore (they were causing bad overlap scoring)
IGNORE_Q = set("""
define simple words write explain give examples example list tell describe in
difference between main components what are why used for
""".split())

def tok(s):
    s = (s or "").lower()
    t = re.findall(r"[a-z0-9_]+", s)
    return [x for x in t if x not in STOP and len(x) > 1]

def detect_intent(q: str) -> str:
    ql = (q or "").lower()
    if any(k in ql for k in ["define", "meaning of", "what is", "difference", "differentiate"]):
        return "DEFINE"
    if any(k in ql for k in ["steps", "how to", "procedure", "create", "open", "click", "right click", "rename", "copy", "paste"]):
        return "PROCEDURE"
    if any(k in ql for k in ["scratch", "blocks", "sprite", "program"]):
        return "CODE"
    return "GENERAL"

def key_terms_from_query(q: str):
    t = [x for x in tok(q) if x not in IGNORE_Q]
    return t

def looks_like_acronym_term(terms):
    # if query is basically "define ICT", keep acronym
    return any(len(t) <= 5 and t.isalpha() for t in terms)

def score_hit(query, text, meta, intent):
    q_terms = key_terms_from_query(query)
    qt = set(q_terms)
    low_text = (text or "").lower()

    # block filters
    bt = (meta.get("block_type") or "").upper()
    if bt in ["WEBLINKS"]:
        return 0

    # ---------- DEFINE (special handling) ----------
    if intent == "DEFINE":
        # If acronym in query (like ICT), prefer expansions / stands for / exact section title
        if looks_like_acronym_term(q_terms):
            # strongest: explicit expansion or stands for
            if ("stands for" in low_text) or ("full form" in low_text) or ("information and communication technology" in low_text):
                return 2
            # strong: section title itself contains expansion/acronym
            sec = (meta.get("section") or "").lower()
            if "information and communication technology" in sec or "(ict)" in sec:
                return 2
            # medium: contains acronym token
            if any(t in low_text for t in q_terms):
                return 1
            return 0

        # Normal define (e.g., data vs information): require query key-terms to appear
        if not qt:
            return 0
        overlap = sum(1 for t in qt if t in low_text) / max(1, len(qt))
        if overlap >= 0.70:
            return 2
        if overlap >= 0.40:
            return 1
        return 0

    # ---------- PROCEDURE ----------
    if intent == "PROCEDURE":
        proc_words = ["click", "open", "select", "right", "folder", "file", "rename", "copy", "paste", "steps"]
        proc_hits = sum(1 for w in proc_words if w in low_text)
        if proc_hits >= 4:
            return 2
        if proc_hits >= 2:
            return 1
        return 0

    # ---------- CODE ----------
    if intent == "CODE":
        if "scratch" in low_text or "block" in low_text or "sprite" in low_text:
            return 2
        return 0

    # ---------- GENERAL ----------
    if not qt:
        return 0
    overlap = sum(1 for t in qt if t in low_text) / max(1, len(qt))
    if overlap >= 0.65:
        return 2
    if overlap >= 0.35:
        return 1
    return 0

def suggest_input_for_row(row, max_marks=3):
    q = row["query"]
    intent = row.get("intent") or detect_intent(q)

    scored = []
    for i, h in enumerate(row["hits"], start=1):
        cid = h.get("chunk_id","")
        meta = {
            "block_type": h.get("block_type",""),
            "chapter": h.get("chapter",""),
            "section": h.get("section","")
        }
        text = id_to_text.get(cid, "")
        sc = score_hit(q, text, meta, intent)
        if sc > 0:
            scored.append((i, sc))

    # For DEFINE: keep only the best 1 (avoid noisy multi-label)
    if intent == "DEFINE":
        scored.sort(key=lambda x: x[1], reverse=True)
        scored = scored[:1]
    else:
        scored.sort(key=lambda x: x[1], reverse=True)
        scored = scored[:max_marks]

    if not scored:
        return "s"
    return " ".join([f"{i}:{sc}" for i, sc in scored])

print("✅ Ready-to-copy label inputs (improved):\n")
for row in eval_rows:
    inp = suggest_input_for_row(row, max_marks=3)
    print(f"{row['query']}  ->  {inp}")


✅ Ready-to-copy label inputs (improved):

Define ICT in simple words.  ->  1:2
Write advantages of ICT in daily life.  ->  1:2 2:1 3:1
Explain OCR and what it does.  ->  1:1 2:1 3:1
What is a computer system? Name its components.  ->  1:1
What are input devices? Give 4 examples.  ->  s
What are output devices? Give 4 examples.  ->  s
Explain CPU in simple words. What does CPU do?  ->  7:2 8:2 9:2
How to create a new folder in Windows? Write steps.  ->  1:2 2:2 3:2
What is Scratch? Why is Scratch used for learning?  ->  1:1
Differentiate between hardware and software.  ->  1:1


In [63]:
# =========================
# CELL 38 — Manual relabel (WITH TEXT PREVIEW + robust input)
# =========================
import os, json, glob

# ---- helpers ----
def load_jsonl(path):
    rows = []
    if not os.path.exists(path):
        return rows
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def save_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

def sanitize_user_input(s: str) -> str:
    # remove stray characters that caused your "``" invalid-format problem
    s = (s or "").strip()
    s = s.replace("`", "")
    s = s.replace("“", '"').replace("”", '"').replace("’", "'")
    return s.strip()

# ---- ensure eval_rows loaded ----
if "EVAL_RUN_PATH" not in globals():
    raise RuntimeError("EVAL_RUN_PATH not found. Run your eval run cells first (CELL 28-30).")

eval_rows = load_jsonl(EVAL_RUN_PATH)

# ---- build id_to_text robustly ----
if "id_to_text" not in globals() or not isinstance(id_to_text, dict) or len(id_to_text) < 5:
    if "docs" in globals() and "metas" in globals():
        id_to_text = {m["chunk_id"]: d for d, m in zip(docs, metas)}
    else:
        # fallback: load from chunks jsonl if docs/metas not present
        chunks_jsonl_guess = os.path.join(blocks_dir, f"{DOC_ID}_chunks.jsonl")
        if not os.path.exists(chunks_jsonl_guess):
            raise RuntimeError("Cannot rebuild id_to_text (docs/metas missing and chunks jsonl not found).")
        tmp = load_jsonl(chunks_jsonl_guess)
        id_to_text = {x["chunk_id"]: x["text"] for x in tmp if "chunk_id" in x and "text" in x}

print("✅ id_to_text size:", len(id_to_text))

# ---- labels file ----
if "EVAL_LABELS_PATH" not in globals():
    EVAL_LABELS_PATH = os.path.join(EVAL_DIR, f"{DOC_ID}_eval_labels.jsonl")

label_rows = load_jsonl(EVAL_LABELS_PATH)
label_map = {r["query"]: r for r in label_rows}

print("\nLabeling instructions:")
print("Format: chunk_id=2,chunk_id=1   (only 1 or 2; omit irrelevant)")
print("Enter = keep existing")
print("Commands: 'skip' | 'clear' | 'stop'\n")

PRINT_CHARS = 750  # ✅ this is the main upgrade vs your current cell

for qi, row in enumerate(eval_rows, start=1):
    q = row["query"]
    intent = row.get("intent", "")
    hits = row.get("hits", [])

    if q not in label_map:
        label_map[q] = {"query": q, "relevance": {}}

    rel = label_map[q].get("relevance", {})
    print("\n" + "="*95)
    print(f"[{qi}/{len(eval_rows)}] QUERY: {q}")
    print("Intent:", intent)
    print("Existing labels:", {k:v for k,v in rel.items() if v in [1,2]})
    print("-"*95)

    # show hits + big snippet
    for rnk, h in enumerate(hits, start=1):
        cid = h.get("chunk_id","")
        ch  = h.get("chapter","")
        sec = h.get("section","")
        bt  = h.get("block_type","")

        cur = rel.get(cid, 0)
        print(f"R{rnk:02d} | {cid} (current={cur}) | {bt} | {ch} | {sec}")

        txt = (id_to_text.get(cid, "") or "").strip()
        if txt:
            snip = txt[:PRINT_CHARS] + ("…" if len(txt) > PRINT_CHARS else "")
            print("-"*90)
            print(snip)
        else:
            print("-"*90)
            print("(no text found for this chunk_id)")
        print()

    user_in = sanitize_user_input(input("Your labels (chunk_id=grade, ...): "))

    if user_in.lower() == "stop":
        break
    if user_in.lower() == "skip":
        continue
    if user_in.lower() == "clear":
        # clear labels for shown hits only
        for h in hits:
            cid = h.get("chunk_id","")
            if cid in rel:
                rel[cid] = 0
        label_map[q]["relevance"] = rel
        continue
    if user_in == "":
        continue

    # parse "cid=2,cid=1"
    updates = {}
    ok = True
    parts = [p.strip() for p in user_in.split(",") if p.strip()]
    for p in parts:
        if "=" not in p:
            ok = False
            break
        cid, val = [x.strip() for x in p.split("=", 1)]
        if val not in ["1","2"]:
            ok = False
            break
        updates[cid] = int(val)

    if not ok:
        print("❌ Invalid format. Example: CS6__000007__7d79d3d7=2,CS6__000028__d1a3c74c=1")
        continue

    # apply updates
    for cid, v in updates.items():
        rel[cid] = v
    label_map[q]["relevance"] = rel
    print("✅ Updated labels")

# save back
out_rows = list(label_map.values())
save_jsonl(EVAL_LABELS_PATH, out_rows)
print("\n✅ Saved labels to:", EVAL_LABELS_PATH)


✅ id_to_text size: 319

Labeling instructions:
Format: chunk_id=2,chunk_id=1   (only 1 or 2; omit irrelevant)
Enter = keep existing
Commands: 'skip' | 'clear' | 'stop'


[1/10] QUERY: Define ICT in simple words.
Intent: DEFINE
Existing labels: {'CS6__000007__7d79d3d7': 2}
-----------------------------------------------------------------------------------------------
R01 | CS6__000007__7d79d3d7 (current=2) | BODY | UNIT 1 — ICT Fundamentals | 1.1 Information and Communication Technology (ICT)
------------------------------------------------------------------------------------------
Information and Communication Technologies (ICT) is defined as a set of computing tools that collectively allow people and organizations to interact in the digital world. ICT is also used to refer to the merging of telephone networks with computer networks through a single link system. ICT is an umbrella term that includes communication devices, television, cell phones, computer and network hardware, satellit

In [37]:
# # =========================
# # DEBUG — Why did labeling show 0 queries?
# # =========================
# import os, json

# def load_jsonl(path):
#     rows = []
#     if not os.path.exists(path):
#         return rows
#     with open(path, "r", encoding="utf-8") as f:
#         for line in f:
#             line = line.strip()
#             if line:
#                 rows.append(json.loads(line))
#     return rows

# print("EVAL_RUN_PATH   :", EVAL_RUN_PATH)
# print("EVAL_LABELS_PATH:", EVAL_LABELS_PATH)

# print("\nFiles exist?")
# print("  eval run   exists:", os.path.exists(EVAL_RUN_PATH), "size:", os.path.getsize(EVAL_RUN_PATH) if os.path.exists(EVAL_RUN_PATH) else 0)
# print("  labels     exists:", os.path.exists(EVAL_LABELS_PATH), "size:", os.path.getsize(EVAL_LABELS_PATH) if os.path.exists(EVAL_LABELS_PATH) else 0)

# eval_rows = load_jsonl(EVAL_RUN_PATH)
# label_rows = load_jsonl(EVAL_LABELS_PATH)

# print("\nCounts:")
# print("  eval_rows:", len(eval_rows))
# print("  label_rows:", len(label_rows))

# already = set(r.get("query") for r in label_rows if isinstance(r, dict) and r.get("query"))
# with_hits = sum(1 for r in eval_rows if (r.get("hits") or []))
# all_labeled = sum(1 for r in eval_rows if r.get("query") in already)

# print("\nEval row quality:")
# print("  rows with hits:", with_hits)
# print("  rows already labeled:", all_labeled)

# if eval_rows:
#     r0 = eval_rows[0]
#     print("\nFirst eval row keys:", list(r0.keys()))
#     print("First query:", r0.get("query"))
#     print("First hits len:", len(r0.get("hits") or []))
#     if r0.get("hits"):
#         print("First hit keys:", list(r0["hits"][0].keys()))


EVAL_RUN_PATH   : /content/drive/MyDrive/FYP/Grade_6/Comp/outputs_cs6/eval/CS6_eval_run_20260207_071453.jsonl
EVAL_LABELS_PATH: /content/drive/MyDrive/FYP/Grade_6/Comp/outputs_cs6/eval/CS6_eval_labels.jsonl

Files exist?
  eval run   exists: True size: 19169
  labels     exists: True size: 3503

Counts:
  eval_rows: 10
  label_rows: 10

Eval row quality:
  rows with hits: 10
  rows already labeled: 10

First eval row keys: ['query', 'intent', 'top_n', 'hits']
First query: Define ICT in simple words.
First hits len: 10
First hit keys: ['chunk_id', 'chapter', 'section', 'block_type', 'src', 'score']


In [36]:
# import os
# p = EVAL_RUN_PATH.replace(".jsonl", "__label_packets.jsonl")
# if os.path.exists(p):
#     os.remove(p)
#     print("Deleted:", p)


Deleted: /content/drive/MyDrive/FYP/Grade_6/Comp/outputs_cs6/eval/CS6_eval_run_20260207_071453__label_packets.jsonl


In [64]:
# =========================
# CELL 34 — Labeling Progress (how many queries actually labeled?)
# =========================
import json, os

def count_labeled(path):
    n_total = 0
    n_labeled = 0
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            r = json.loads(line)
            n_total += 1
            rel = r.get("relevance", {})
            if any(int(v) > 0 for v in rel.values()):
                n_labeled += 1
    return n_total, n_labeled

total_q, labeled_q = count_labeled(EVAL_LABELS_PATH)
print("Total queries:", total_q)
print("Queries with >=1 relevant chunk labeled:", labeled_q)
print("Labels file:", EVAL_LABELS_PATH)


Total queries: 10
Queries with >=1 relevant chunk labeled: 10
Labels file: /content/drive/MyDrive/FYP/Grade_6/Comp/outputs_cs6/eval/CS6_eval_labels.jsonl


In [65]:
# =========================
# CELL 35 — Compute Retrieval Metrics (Recall/Precision/MRR/NDCG/ForbiddenTypeRate)
# =========================
import json, math

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(x) for x in f]

def normalize_cid(cid):
    return str(cid).strip().lower()

def dcg(rels):
    s = 0.0
    for i, rel in enumerate(rels, start=1):
        s += (2**rel - 1) / math.log2(i + 1)
    return s

eval_rows = load_jsonl(EVAL_RUN_PATH)
label_rows = load_jsonl(EVAL_LABELS_PATH)

labels = {}
for r in label_rows:
    q = r["query"]
    rel = r.get("relevance", {}) or {}
    labels[q] = {normalize_cid(cid): int(g) for cid, g in rel.items()}

FORBIDDEN_TYPES = {"SLO", "WEBLINKS"}

def compute_metrics_at_k(k):
    n_labeled = 0
    recall_sum = 0.0
    prec_sum = 0.0
    mrr_sum = 0.0
    ndcg_sum = 0.0
    forbid_sum = 0.0

    for row in eval_rows:
        q = row["query"]
        if q not in labels:
            continue

        rel_map = labels[q]
        relevant = {cid for cid, g in rel_map.items() if int(g) > 0}
        if not relevant:
            continue

        hits = row.get("hits", [])[:k]
        ranked_ids = [normalize_cid(h.get("chunk_id", "")) for h in hits]

        # Recall@k (binary)
        hit_any = any(cid in relevant for cid in ranked_ids)
        recall_sum += 1.0 if hit_any else 0.0

        # Precision@k (binary)
        rel_in_topk = sum(1 for cid in ranked_ids if cid in relevant)
        prec_sum += rel_in_topk / float(k)

        # MRR@k (binary)
        rr = 0.0
        for i, cid in enumerate(ranked_ids, start=1):
            if cid in relevant:
                rr = 1.0 / i
                break
        mrr_sum += rr

        # NDCG@k (graded 0/1/2)
        rels_ranked = [int(rel_map.get(cid, 0)) for cid in ranked_ids]
        ideal_rels = sorted([int(g) for g in rel_map.values() if int(g) > 0], reverse=True)[:k]
        dcg_val = dcg(rels_ranked)
        idcg_val = dcg(ideal_rels) if ideal_rels else 0.0
        ndcg_sum += (dcg_val / idcg_val) if idcg_val > 0 else 0.0

        # ForbiddenTypeRate@k
        forb = 0
        for h in hits:
            bt = (h.get("block_type") or "").strip().upper()
            if bt in FORBIDDEN_TYPES:
                forb += 1
        forbid_sum += forb / float(k)

        n_labeled += 1

    if n_labeled == 0:
        return None

    return {
        "n_labeled": n_labeled,
        f"Recall@{k}": recall_sum / n_labeled,
        f"Precision@{k}": prec_sum / n_labeled,
        f"MRR@{k}": mrr_sum / n_labeled,
        f"NDCG@{k}": ndcg_sum / n_labeled,
        f"ForbiddenTypeRate@{k}": forbid_sum / n_labeled,
    }

for K in [3, 5, 10]:
    m = compute_metrics_at_k(K)
    print("\n" + "="*70)
    if m is None:
        print(f"❌ No labeled queries found for K={K}. Label at least 1 query with score 1/2.")
        continue
    for kk, vv in m.items():
        if isinstance(vv, float):
            print(f"{kk}: {vv:.4f}")
        else:
            print(f"{kk}: {vv}")



n_labeled: 10
Recall@3: 1.0000
Precision@3: 0.5000
MRR@3: 0.8500
NDCG@3: 0.7644
ForbiddenTypeRate@3: 0.0000

n_labeled: 10
Recall@5: 1.0000
Precision@5: 0.3400
MRR@5: 0.8500
NDCG@5: 0.7956
ForbiddenTypeRate@5: 0.0000

n_labeled: 10
Recall@10: 1.0000
Precision@10: 0.2200
MRR@10: 0.8500
NDCG@10: 0.8583
ForbiddenTypeRate@10: 0.0000


In [ ]:
# # =========================
# # CELL 36 — Show Failures (queries where Recall@5 failed)
# # =========================
# K = 5

# fails = []
# for row in eval_rows:
#     q = row["query"]
#     if q not in labels:
#         continue
#     rel_map = labels[q]
#     relevant = {cid for cid, g in rel_map.items() if int(g) > 0}
#     if not relevant:
#         continue

#     ranked = row.get("hits", [])[:K]
#     ranked_ids = [normalize_cid(h.get("chunk_id","")) for h in ranked]
#     if not any(cid in relevant for cid in ranked_ids):
#         fails.append((q, ranked))

# print(f"Recall@{K} failures:", len(fails))

# for q, ranked in fails[:10]:
#     print("\n" + "-"*100)
#     print("QUERY:", q)
#     for i, h in enumerate(ranked, start=1):
#         print(f"{i}. {h.get('chunk_id','')} | {h.get('block_type','')} | {h.get('chapter','')} | {h.get('section','')}")


Recall@5 failures: 1

----------------------------------------------------------------------------------------------------
QUERY: Explain CPU in simple words. What does CPU do?
1. CS6__000060__e1835166 | BODY | UNIT 2 — Components of Computer System | 2.5 Central Processing Unit (CPU)
2. CS6__000039__52bc83d6 | BODY | UNIT 2 — Components of Computer System | 2.1 Basic Components of Computer
3. CS6__000038__96f4f0e5 | BODY | UNIT 2 — Components of Computer System | Introduction
4. CS6__000014__5a09a4df | BODY | UNIT 1 — ICT Fundamentals | 1.3 Software
5. CS6__000075__0918bdfb | BODY | UNIT 2 — Components of Computer System | 2.5 Central Processing Unit (CPU)


In [44]:
# =========================
# CELL 37 — (Optional) Run Full RAG Answer on Eval Queries (quick sanity)
# =========================
# Requires you already loaded the LLM cells (tokenizer/model + llm_generate + rag_answer).
# If rag_answer exists, this will print answers for the 10 eval queries.

if "rag_answer" not in globals():
    print("❌ rag_answer not found. Run your LLM cells first.")
else:
    for q in EVAL_QUERIES:
        r = rag_answer(q, top_k=6)
        print("\n" + "="*110)
        print("Q:", q)
        print("A:", r["answer"][:900])
        print("Sources:", r["sources"])



Q: Define ICT in simple words.
A: system

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a helpful tutor. Follow the rules strictly.user

You are a Grade 6 Computer Science tutor.

RULES (VERY IMPORTANT):
- Use ONLY the provided CONTEXT (textbook chunks).
- If the context is not enough, say exactly: "I don't have that information in the book context."
- Explain in simple words.
- If the question asks steps, answer in numbered steps.
- End with: Sources: [chunk_id1, chunk_id2,...]

QUESTION:
Define ICT in simple words.

CONTEXT:
[1] chunk_id=CS6__000007__7d79d3d7 | BODY | UNIT 1 — ICT Fundamentals | 1.1 Information and Communication Technology (ICT)
Information and Communication Technologies (ICT) is defined as a set of computing tools that collectively allow people and organizations to interact in the digital world. ICT is also used to refer to the merging of telephone networks with computer netwo
Sources: ['CS6__000007__7d79d3d7', 'CS6__000028__d1a3c74c', 'CS6

In [ ]:
# # =========================
# # CELL 36 — Fix detect_intent (examples != CODE)
# # =========================
# def detect_intent(q: str) -> str:
#     ql = (q or "").lower()

#     # DEFINE
#     if any(k in ql for k in ["define", "what is", "meaning of", "difference", "differentiate"]):
#         return "DEFINE"

#     # PROCEDURE
#     if any(k in ql for k in ["steps", "how to", "procedure", "create", "open", "click", "right click"]):
#         return "PROCEDURE"

#     # EXERCISE
#     if any(k in ql for k in ["exercise", "mcq", "fill in", "tick", "briefly", "question"]):
#         return "EXERCISE"

#     # CODE (only real coding words)
#     if any(k in ql for k in ["code", "program", "scratch", "python", "c++", "algorithm"]):
#         return "CODE"

#     return "GENERAL"

# print("✅ detect_intent patched")


✅ detect_intent patched


In [ ]:
# # =========================
# # CELL 37 — Clean LLM output extractor
# # =========================
# import re

# def extract_answer_only(generated_text: str) -> str:
#     """
#     Tries to return only the actual answer part.
#     Works even if decoded text contains 'system', 'user', etc.
#     """
#     t = (generated_text or "").strip()

#     # If our prompt includes "ANSWER:", take everything after the LAST occurrence
#     if "ANSWER:" in t:
#         t = t.split("ANSWER:")[-1].strip()

#     # Remove leading role markers if present
#     t = re.sub(r"^\s*(system|user|assistant)\s*\n+", "", t, flags=re.IGNORECASE)

#     # Sometimes "assistant" appears inline
#     t = re.sub(r"(?i)^assistant\s*", "", t).strip()

#     return t

# @torch.inference_mode()
# def llm_generate(prompt: str, max_new_tokens=220, temperature=0.2, top_p=0.9):
#     text_in = format_chat_prompt(prompt)

#     inputs = tokenizer(text_in, return_tensors="pt", padding=True)
#     inputs = {k: v.to(model.device) for k, v in inputs.items()}

#     out = model.generate(
#         **inputs,
#         max_new_tokens=max_new_tokens,
#         do_sample=(temperature > 0),
#         temperature=temperature,
#         top_p=top_p,
#         pad_token_id=tokenizer.pad_token_id,
#         eos_token_id=tokenizer.eos_token_id
#     )

#     decoded = tokenizer.decode(out[0], skip_special_tokens=True)

#     # Remove echoed prompt if present
#     if decoded.startswith(text_in):
#         decoded = decoded[len(text_in):].strip()

#     return extract_answer_only(decoded)

# print("✅ llm_generate patched (clean output)")


✅ llm_generate patched (clean output)


In [ ]:
# # =========================
# # CELL 38 — Re-test RAG output (clean)
# # =========================
# q1 = "Define ICT in simple words."
# r1 = rag_answer(q1, top_k=6)
# print("Q:", q1)
# print(r1["answer"])
# print("Sources:", r1["sources"])


Q: Define ICT in simple words.
ICT stands for Information and Communication Technology. It is a set of computing tools that helps people and organizations interact in the digital world. ICT includes devices, networks, and services like computers, phones, television, and the internet.

Sources: CS6__000007__7d79d3d7, CS6__000028__d1a3c74c, CS6__000027__af3505c2, CS6__000022__a214acc5, CS6__000023__418b7cf6, CS6__000029__acddffa4
Sources: ['CS6__000007__7d79d3d7', 'CS6__000028__d1a3c74c', 'CS6__000027__af3505c2', 'CS6__000022__a214acc5', 'CS6__000023__418b7cf6', 'CS6__000029__acddffa4']


In [ ]:
# # =========================
# # CELL 39 — Find unlabeled queries
# # =========================
# import json

# def load_jsonl(path):
#     with open(path, "r", encoding="utf-8") as f:
#         return [json.loads(x) for x in f]

# label_rows = load_jsonl(EVAL_LABELS_PATH)

# unlabeled = []
# for r in label_rows:
#     rel = r.get("relevance", {}) or {}
#     if not any(int(v) > 0 for v in rel.values()):
#         unlabeled.append(r["query"])

# print("✅ Unlabeled queries:", len(unlabeled))
# for q in unlabeled:
#     print("-", q)


✅ Unlabeled queries: 2
- What are input devices? Give 4 examples.
- What are output devices? Give 4 examples.


In [ ]:
# # =========================
# # CELL 40 — Continue labeling ONLY unlabeled queries
# # =========================
# import os, json

# def save_jsonl(path, rows):
#     with open(path, "w", encoding="utf-8") as f:
#         for r in rows:
#             f.write(json.dumps(r, ensure_ascii=False) + "\n")

# eval_rows = load_jsonl(EVAL_RUN_PATH)
# label_rows = load_jsonl(EVAL_LABELS_PATH)

# # Build helper map
# label_map = {r["query"]: r for r in label_rows}

# def show_query_hits(query, k=10, chars=650):
#     row = next((r for r in eval_rows if r["query"] == query), None)
#     if row is None:
#         print("❌ Query not found")
#         return None

#     print("\n" + "="*110)
#     print("QUERY:", row["query"])
#     print("INTENT:", row.get("intent"))
#     print("="*110)

#     for i, h in enumerate(row["hits"][:k], start=1):
#         cid = h.get("chunk_id","")
#         ch  = h.get("chapter","")
#         sec = h.get("section","")
#         bt  = h.get("block_type","")
#         txt = id_to_text.get(cid, "")
#         snip = (txt[:chars] + "…") if len(txt) > chars else txt
#         print(f"\n{i}. (current={label_map[query]['relevance'].get(cid,0)}) {cid} | {bt} | {ch} | {sec}")
#         print("-"*90)
#         print(snip)
#     return row

# # label only those with no positive relevance
# targets = []
# for r in label_rows:
#     rel = r.get("relevance", {}) or {}
#     if not any(int(v) > 0 for v in rel.values()):
#         targets.append(r["query"])

# print("Targets to label:", targets)

# for q in targets:
#     row = show_query_hits(q, k=10, chars=650)
#     if row is None:
#         continue

#     sug = "s"
#     print("\nType i:score pairs like 2:2 5:1 (0/1/2). Example: 2:2 means hit #2 is highly relevant.")
#     user_in = input("Your input (or 's' skip | 'r' reset | 'q' quit): ").strip().lower()

#     if user_in == "q":
#         print("Stopping labeling.")
#         break
#     if user_in == "s":
#         continue
#     if user_in == "r":
#         # reset all to 0
#         label_map[q]["relevance"] = {h["chunk_id"]: 0 for h in row.get("hits", []) if h.get("chunk_id")}
#         save_jsonl(EVAL_LABELS_PATH, list(label_map.values()))
#         print("✅ Reset saved.")
#         continue

#     # parse pairs
#     pairs = user_in.split()
#     for p in pairs:
#         i_s, sc_s = p.split(":")
#         i = int(i_s)
#         sc = int(sc_s)
#         if 1 <= i <= len(row["hits"]):
#             cid = row["hits"][i-1]["chunk_id"]
#             label_map[q]["relevance"][cid] = sc

#     save_jsonl(EVAL_LABELS_PATH, list(label_map.values()))
#     print("✅ Saved labels for:", q)

# print("✅ Done labeling remaining queries.")
# print("Labels file:", EVAL_LABELS_PATH)


Targets to label: ['What are input devices? Give 4 examples.', 'What are output devices? Give 4 examples.']

QUERY: What are input devices? Give 4 examples.
INTENT: CODE

1. (current=0) CS6__000015__c0f65c90 | BODY | UNIT 1 — ICT Fundamentals | 1.3 Software
------------------------------------------------------------------------------------------
Peripheral devices are hardware used for input, secondary storage and display etc. Peripherals are attached to the system unit through a hardware interface. We will discuss peripherals in detail later in this book.

2. (current=0) CS6__000076__145ffc21 | BODY | UNIT 2 — Components of Computer System | 2.6 Working of Computer System
------------------------------------------------------------------------------------------
Computer is an electronic machine which processes the data (input) to produce the desired information (output), and saves data (storage) according to the given instructions. A computer works by combining input, storage, proces

In [ ]:
# =========================
# CELL 41 — Recompute metrics after completing labels
# =========================
# (re-run your CELL 35 code block exactly)
for K in [3, 5, 10]:
    m = compute_metrics_at_k(K)
    print("\n" + "="*70)
    if m is None:
        print(f"❌ No labeled queries found for K={K}.")
        continue
    for kk, vv in m.items():
        if isinstance(vv, float):
            print(f"{kk}: {vv:.4f}")
        else:
            print(f"{kk}: {vv}")



n_labeled: 10
Recall@3: 0.9000
Precision@3: 0.5333
MRR@3: 0.8500
NDCG@3: 0.8107
ForbiddenTypeRate@3: 0.0000

n_labeled: 10
Recall@5: 0.9000
Precision@5: 0.3600
MRR@5: 0.8500
NDCG@5: 0.8135
ForbiddenTypeRate@5: 0.0000

n_labeled: 10
Recall@10: 1.0000
Precision@10: 0.2500
MRR@10: 0.8643
NDCG@10: 0.8918
ForbiddenTypeRate@10: 0.0000


In [ ]:
# =========================
# CELL 42 — Show failures at K=3
# =========================
label_rows = load_jsonl(EVAL_LABELS_PATH)
labels = {r["query"]: {normalize_cid(cid): int(g) for cid, g in (r.get("relevance", {}) or {}).items()}
          for r in label_rows}

def show_failures_at_k(k=3):
    bad = []
    for row in eval_rows:
        q = row["query"]
        rel_map = labels.get(q, {})
        relevant = {cid for cid, g in rel_map.items() if g > 0}
        if not relevant:
            continue
        ranked = [normalize_cid(h.get("chunk_id","")) for h in row.get("hits", [])[:k]]
        if not any(cid in relevant for cid in ranked):
            bad.append(q)

    print(f"Recall@{k} failures:", len(bad))
    for q in bad:
        print("\n---", q)
        preview_query(q, k=10, chars=450)

show_failures_at_k(3)


Recall@3 failures: 1

--- Explain CPU in simple words. What does CPU do?

QUERY: Explain CPU in simple words. What does CPU do?
INTENT: GENERAL

1. CS6__000060__e1835166 | BODY | UNIT 2 — Components of Computer System | 2.5 Central Processing Unit (CPU)
------------------------------------------------------------------------------------------
The Central Processing Unit is the core of any computer system. It comprises of three major components which have been discussed below:
* Memory Unit
* Control Unit
* Arithmetic and Logical Unit

All these three units are elements of CPU and together help in the working and processing of data. It is also known as the "Brain of Computer" and no action can be taken by a system without the execution and permission of the Central Processing Unit. Mai…

2. CS6__000039__52bc83d6 | BODY | UNIT 2 — Components of Computer System | 2.1 Basic Components of Computer
------------------------------------------------------------------------------------------
Com